<b><center><font size="4">TriSem-LLM <br /> Multilingual semantic pipeline for systematic reviews</font></center></b>
<center><font size="2">
Authors: <br /> <b>Tassia Carvalho, Sandra Jardim</b><br />Smart Cities Research Center, Polytechnic University of Tomar</b><br />(tassia.scarvalho@ipt.pt; sandra.jardim@ipt.pt)</font></center>
<br />
<div style="text-align: center;">
    <font size="1">This work was funded by the <b>Project RA3I - Rock Art Anal- ysis with Artificial Intelligence</b>, with code COMPETE2030-
FEDER-00536800, co-financed by Portugal 2030.</font><br>
</div>

*Extract data from PDFs*

This section implements the first stage of the TriSem-LLM pipeline, which focuses on the automated extraction of structured information from scientific articles in PDF format. For each document, the pipeline extracts the title, abstract, publication source, and full text. The extracted content is cleaned and organized into a structured tabular format (Pandas DataFrame), which is subsequently exported as a .csv file for downstream processing.

In addition, the full text of each article is stored as an individual .txt file to support subsequent semantic analysis at both abstract and full-text levels. This step ensures the standardization and availability of textual data required for the multi-stage screening methodology described in the paper.

*Imports and Configuration*

In [ ]:
pip install sentence-transformers

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install pymupdf

In [7]:
import os
import fitz  # PyMuPDF
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import time
import webbrowser
from sentence_transformers import SentenceTransformer, util
from IPython.display import display
from difflib import SequenceMatcher
from transformers import AutoTokenizer

USE_LOCAL_PDFS = False  # Set to True only if PDFs are available locally

# Paths
pdf_folder = 'data/pdfs'  # optional, used only if USE_LOCAL_PDFS=True
fulltext_folder = 'data/full_texts'  # optional
output_csv = '/content/data/extracted_articles.csv'

# Ensure output directory exists
if USE_LOCAL_PDFS:
    os.makedirs(fulltext_folder, exist_ok=True)

 *Text Cleaning and PDF Text Extraction*

In [8]:
# This block defines utility functions for cleaning extracted text and retrieving full textual content from PDF documents, forming the basis for downstream analysis.

# Clean text by removing line breaks, extra spaces, etc.
def clean_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' \n')


# Reads the PDF and returns cleaned full text and per-page content
def extract_pdf_text(path):
    if not USE_LOCAL_PDFS:
        raise RuntimeError("PDF extraction is disabled. Set USE_LOCAL_PDFS = True to enable this feature.")

    doc = fitz.open(path)
    full_text = ''
    pages = []
    for page in doc:
        text = page.get_text()
        pages.append(text)
        full_text += text

    return clean_text(full_text), pages

*PDF extraction functions are included for completeness but are disabled by default.*

*To enable full-text processing, users must provide local access to the PDF files and set USE_LOCAL_PDFS = True.*

*Title Extraction from PDF Documents*

In [9]:
# This block implements a heuristic-based approach to extract the article title from the first page of each PDF, based on font size and positional features.
# Additional filtering rules are applied to remove non-title elements.

def extract_title(pdf_path):

    if not USE_LOCAL_PDFS:
        raise RuntimeError("Title extraction requires local PDFs. Set USE_LOCAL_PDFS = True to enable this feature.")

    suspicious_patterns = [
        "journal", "revista", "editorial", "cover", "contents", "introduction",
        "biographies", "foreword", "abstract", "résumé", "resumen",
        "to cite this article", "keywords", "introduction", "conference",
        "materials and methods", "references"
    ]

    ignored_titles = [
        "title not identified",
        "to link to this article:",
        "an update to this article is included at the end",
        "digital applications in archaeology and cultural heritage",
        "2023 ieee international conference on advanced learning technologies",
        "2020 IEEE 24th International Enterprise Distributed Object Computing Conference (EDOC)",
        "proceedings of the sixth international conference on machine learning and cybernetics, hong kong, 19-22 august 2007",
        "2012 ieee 24th international conference on tools with artificial intelligence",
        "2024 international conference on cyberworlds (cw)",
        "proceedings of the 6 th german microwave conference"
    ]

    doc = fitz.open(pdf_path)
    page = doc[0]
    blocks = page.get_text("dict")["blocks"]

    candidates = []

    for block in blocks:
        block_text = []
        block_size = []
        block_y = []
        for line in block.get("lines", []):
            line_text = " ".join(span["text"].strip() for span in line.get("spans", []))
            line_size = max(span["size"] for span in line.get("spans", []))
            line_y = min(span["bbox"][1] for span in line.get("spans", []))

            # Filters
            if len(line_text.split()) < 2:
                continue
            if any(p in line_text.lower() for p in suspicious_patterns):
                continue
            if line_size < 10:
                continue

            block_text.append(line_text)
            block_size.append(line_size)
            block_y.append(line_y)

        if block_text:
            full_block_text = " ".join(block_text).strip()
            full_block_text_lower = full_block_text.lower()
            if full_block_text_lower in ignored_titles:
                continue

            candidates.append({
                "text": full_block_text,
                "avg_size": sum(block_size)/len(block_size),
                "min_y": min(block_y),
                "n_words": len(full_block_text.split())
            })

    if not candidates:
        return "Title not identified"

    # Sort by vertical position
    candidates.sort(key=lambda x: x["min_y"])

    # Filter candidates with at least 6 words
    candidates = [c for c in candidates if c["n_words"] >= 6]

    # Return first valid block
    if candidates:
        return candidates[0]["text"]

    return "Title not identified"

*Multilingual Abstract Extraction*

In [10]:
# This block extracts the abstract from each document using multilingual regular expression patterns (English, Spanish, and French),
# complemented by fallback heuristic strategies when explicit labels are not detected.

def extract_abstract(text, pages):

    if not USE_LOCAL_PDFS:
        raise RuntimeError("Abstract extraction from PDF pages requires local PDFs. Set USE_LOCAL_PDFS = True.")

    # Limit search to first 3 pages
    text_to_search = "\n".join(pages[:3])

    # Regex for English, Spanish, and French
    patterns = [
        r"(?ims)\b(?:abstract|a\s*b\s*s\s*t\s*r\s*a\s*c\s*t)\b[:\s–—-]*(.*?)(?=\n(?:\d+\.\s|index terms|keywords|introduction|I\.\s*INTRODUCTION|INDEX TERMS|KEYWORDS|INTRODUCTION))",
        r"(?ims)\b(?:resumen|r\s*e\s*s\s*u\s*m\s*e\s*n)\b[:\s–—-]*(.*?)(?=\n(?:\d+\.\s|palabras clave|introducción|INDEX TERMS|PALABRAS CLAVE|INTRODUCCIÓN))",
        r"(?ims)\b(?:résumé|r\s*é\s*s\s*u\s*m\s*é)\b[:\s–—-]*(.*?)(?=\n(?:\d+\.\s|mots[-\s]clés|introduction|INDEX TERMS|MOTS[-\s]CLÉS|INTRODUCTION))"
    ]

    for pattern in patterns:
        match = re.search(pattern, text_to_search)
        if match:
            return clean_text(match.group(1)), "label"

    # Heuristic using common phrases
    common_phrases = [
        'this paper', 'we present', 'the study', 'in this work',
        'this research', 'our findings', 'the results suggest',
        'este artículo', 'este trabajo', 'en esta investigación',
        'nuestros hallazgos', 'los resultados sugieren',
        'cet article', 'cette étude', 'nous présentons',
        'cette recherche', 'nos résultats suggèrent'
    ]

    for i in range(min(2, len(pages))):
        blocks = re.split(r'\n\s*\n', pages[i])
        for block in blocks:
            if len(block.split()) >= 50 and any(phrase in block.lower() for phrase in common_phrases):
                return clean_text(block), "heuristic"

    # Fallback: block before Keywords
    for i in range(min(3, len(pages))):
        keywords_match = re.search(r"(?i)\nkeywords[:\s]", pages[i])
        if keywords_match:
            before_keywords = pages[i][:keywords_match.start()]
            if len(before_keywords.split()) >= 40:
                return clean_text(before_keywords), "before_keywords"

    # Fallback: first large block
    blocks = re.split(r'\n\s*\n', pages[0])
    candidates = [b for b in blocks if len(b.split()) >= 40]
    if candidates:
        return clean_text(candidates[0]), "first_large_block"

    # Fallback: until introduction
    for i in range(min(3, len(pages))):
        intro_match = re.search(
            r"(?:\n|\s)(\d+\.\s*INTRODUCTION|I\.\s*INTRODUCTION)",
            pages[i],
            re.I
        )
        if intro_match:
            before_intro = pages[i][:intro_match.start()]
            return clean_text(before_intro), "until_introduction"

    return "", "notfound"

*Publication Source Extraction*

In [11]:
# This block extracts the publication or journal source from each document
# using multilingual regular expression patterns applied to the full text.

def extract_publication(text):

    if text is None or len(text.strip()) == 0:
        return "Publication not identified"

    patterns = [
        # English/Spanish/French mixed
        r"(?i)(?:journal|revista|revue|conference|congreso|conférence|proceedings)\s*[:\-]?\s*(.{10,100}?)\s{2,}",
        r"(?i)(published in|publicado en|publié dans)\s+(.{10,100}?)\s{2,}",
        r"(?i)\b(journal.*?|revue.*?|revista.*?)\b"
    ]

    for pattern in patterns:
        result = re.search(pattern, text, re.I | re.S)
        if result:
            return clean_text(result.group(result.lastindex or 1))

    return "Publication not identified"

*Document Processing and Metadata Extraction*

In [ ]:
# This block iterates over all PDF documents, applies the extraction
# functions (title, abstract, publication, and full text), and structures
# the results into a list for subsequent DataFrame creation. The full text
# of each document is also saved as a separate .txt file.

extracted_data = []

if USE_LOCAL_PDFS:
    print("🔹 Extracting from local PDFs...")

    for root, _, files in os.walk(pdf_folder):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_path = os.path.join(root, file)

                full_text, pages_text = extract_pdf_text(pdf_path)
                abstract_text, abstract_source = extract_abstract(full_text, pages_text)

                subfolder_name = os.path.basename(root)

                txt_filename = file.replace(".pdf", ".txt")
                txt_path = os.path.join(fulltext_folder, txt_filename)

                with open(txt_path, "w", encoding="utf-8") as ft:
                    ft.write(full_text)

                extracted_data.append({
                    "file_name": file,
                    "title": extract_title(pdf_path),
                    "abstract": abstract_text,
                    "abstractSource": abstract_source,
                    "publication": extract_publication(full_text),
                    "repository": subfolder_name
                })

    # Save extracted data
    df = pd.DataFrame(extracted_data)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print("✅ Extraction completed from PDFs!")

else:
    print("📄 Loading pre-extracted dataset...")
    df = pd.read_csv(output_csv)

display(df.head())

*Export to HTML for Interactive Inspection*

In [ ]:
# This block exports the extracted dataset to an HTML table format,
# enabling interactive inspection of the results.

if USE_LOCAL_PDFS:
    print("🔹 Generating HTML with local PDF links...")

    df_html = df.copy()

    df_html["filename"] = df_html.apply(
        lambda row: f'<a href="file:///{os.path.abspath(os.path.join(pdf_folder, row["repository"], row["filename"]))}" target="_blank">{row["filename"]}</a>'
        if pd.notna(row["filename"]) else "",
        axis=1
    )

else:
    print("🔹 Generating HTML without local file links...")

    df_html = df.copy()
    df_html["filename"] = df_html["filename"].astype(str)

# Convert to HTML
html_output = df_html.to_html(
    index=False,
    escape=False,
    justify="center"
)

# Save file
output_file = "data/extracted_articles.html"
os.makedirs("data", exist_ok=True)  # ✅ important safety fix

with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"✅ HTML table exported to: {output_file}")

*Metadata Refinement: Title Correction via LLM*

In [ ]:
# This block identifies potentially incorrect or generic titles (e.g., journal
# names or placeholder text) extracted during the initial parsing stage. A large
# language model (LLM) is then used to generate a more accurate title based on
# the content of the first page of each document. The corrected titles are stored
# in a new column ("title_llm"), preserving the original values for traceability.

import os
import time
import requests
import pandas as pd
import re

# =========================
# CONFIGURATION
# =========================

USE_LLM = False  # ⚠️ IMPORTANT: keep False for reproducibility

API_KEY = os.getenv("TOGETHER_API_KEY")
MODEL = "mistralai/Mixtral-8x7B-Instruct-v0.1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# =========================
# TITLE VALIDATION
# =========================

def is_suspicious_title(title):
    if not isinstance(title, str) or len(title.strip()) < 15:
        return True

    suspicious_patterns = [
        "journal", "revista", "editorial", "cover", "contents", "introduction",
        "biographies", "foreword", "to cite this article",
        "abstract", "résumé", "resumen",
        "keywords", "materials and methods", "references"
    ]

    title_lower = title.strip().lower()

    return any(p in title_lower for p in suspicious_patterns)


# =========================
# LLM UTILITIES
# =========================

def clean_llm_response(response):
    if not response:
        return None

    response = re.sub(r"```.*?```", "", response, flags=re.DOTALL)
    response = re.sub(r"(?i)^.*?(title of the article is|the title is):?", "", response)

    lines = [line.strip() for line in response.split('\n') if len(line.strip().split()) > 3]
    return lines[0] if lines else response.strip()


def generate_title_with_llm(text, retries=3):
    if not API_KEY:
        print("⚠️ No API key provided. Skipping LLM.")
        return None

    prompt = (
        "Extract ONLY the title of the scientific article.\n\n"
        f"{text[:3000]}"
    )

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3,
        "max_tokens": 100
    }

    for attempt in range(retries):
        try:
            response = requests.post(
                "https://api.together.xyz/v1/chat/completions",
                headers=HEADERS,
                json=payload,
                timeout=30
            )

            if response.status_code == 200:
                raw_title = response.json()["choices"][0]["message"]["content"]
                return clean_llm_response(raw_title)

            time.sleep(5)

        except Exception as e:
            print(f"❌ Error: {e}")
            time.sleep(5)

    return None


# =========================
# MAIN FUNCTION
# =========================

def correct_titles_with_llm(df):

    if not USE_LLM:
        print("⏭️ Skipping LLM title correction (disabled).")
        return df

    df = df.copy()

    if "title_llm" not in df.columns:
        df["title_llm"] = None

    suspicious_df = df[df["title"].apply(is_suspicious_title)]

    print(f"🔍 {len(suspicious_df)} suspicious titles detected...")

    for idx, row in suspicious_df.iterrows():

        text = row.get("abstract", "")

        if not text:
            continue

        new_title = generate_title_with_llm(text)

        if new_title:
            df.at[idx, "title_llm"] = new_title

        time.sleep(1)

    print("✅ Title correction finished.")
    return df


# =========================
# EXECUTION
# =========================

df = correct_titles_with_llm(df)

*Duplicate Detection Based on Final Titles*

In [ ]:
# This block identifies potential duplicate articles based on the final
# title ("title_final"), which prioritizes the LLM-corrected title when available.
# String similarity is used to detect highly similar titles, and duplicate pairs
# are generated to support manual verification. The results are exported as a CSV file.

# Ensure title_llm exists
if "title_llm" not in df.columns:
    df["title_llm"] = None

# Final title (LLM corrected if available)
df["title_final"] = df["title_llm"].fillna(df["title"])

pairs = []
seen_pairs = set()

# Convert to list (faster than iterrows twice)
rows = df.to_dict("records")

for i in range(len(rows)):
    row_i = rows[i]

    title_i = row_i["title_final"]
    file_i = row_i["filename"]
    repo_i = row_i.get("repository", "")

    for j in range(i + 1, len(rows)):
        row_j = rows[j]

        title_j = row_j["title_final"]
        file_j = row_j["filename"]
        repo_j = row_j.get("repository", "")

        # Compute similarity
        similarity = SequenceMatcher(None, title_i, title_j).ratio()

        if similarity >= 0.95:
            key = tuple(sorted((file_i, file_j)))

            if key not in seen_pairs:
                pairs.append({
                    "file_A": file_i,
                    "repository_A": repo_i,
                    "title_A": title_i,
                    "file_B": file_j,
                    "repository_B": repo_j,
                    "title_B": title_j,
                    "similarity": round(similarity, 3)
                })
                seen_pairs.add(key)

# Create DataFrame
df_pairs = pd.DataFrame(pairs)

# Show results
print("🔁 Duplicate file pairs with similarity ≥ 0.95:")
display(df_pairs)
print(f"🔢 Total duplicate pairs found: {len(pairs)}")

# Save CSV (IMPORTANT: consistent name)
output_duplicates = "data/duplicated_file_pairs.csv"
df_pairs.to_csv(output_duplicates, index=False, encoding="utf-8-sig")

print(f"✅ Duplicate pairs saved to: {output_duplicates}")

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(
        df_pairs.sort_values(by=["repository_B", "similarity"], ascending=[True, False])
    )


<hr>

*Semantic Relevance Assessment*

*Next blocks evaluates the semantic relevance of each article with respect to the research question using embedding-based similarity. Abstracts and full texts are encoded into a shared semantic vector space, and their similarity to a multilingual query (English, Spanish, and French) is computed.*

*This enables the identification of relevant articles based on both abstract-level and full-text semantic alignment.*

In [21]:
# =========================
# Abstract Text Normalization
# =========================
# Normalize abstract text prior to embedding by removing noise and formatting artifacts.

df["abstract_clean"] = (
    df["abstract"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# =========================
# KEYWORD SIGNAL (NO FILTERING)
# =========================
# Keyword matching is used as an auxiliary signal and does NOT trigger exclusion

def compute_keyword_signal(df):

    def contains_keywords(text):
        text = text.lower()

        technical_terms = [
            'digital imaging', 'image processing', 'computer vision', 'artificial intelligence',
            'machine learning', 'deep learning', 'pattern recognition', 'image enhancement',
            'segmentation', 'classification', 'pca', 'dstretch', 'decorrelation stretch',
            'photogrammetry', '3d scanning', 'rti', 'cnn', 'neural network', 'semantic segmentation',
            'procesamiento de imágenes', 'visión por computador', 'inteligencia artificial',
            'aprendizaje automático', 'aprendizaje profundo', 'mejora de imágenes', 'segmentación',
            'clasificación', 'fotogrametría', 'red neuronal', 'redes convolucionales',
            'traitement d\'images', 'vision par ordinateur', 'intelligence artificielle',
            'apprentissage automatique', 'apprentissage profond', 'amélioration d\'image',
            'photogrammétrie', 'réseau neuronal', 'réseaux convolutifs',
            'digital image processing', 'image segmentation', 'automatic classification',
            'semantic analysis', 'neural networks', 'deep neural networks',
            'feature extraction', 'edge detection', 'histogram equalization',
            'principal component analysis', 'machine perception', 'object detection',
            'image recognition', 'unsupervised learning', 'supervised learning',
            'clustering', 'preprocessing', 'pipeline', 'convolutional neural networks',
            'deep learning models', 'U-Net', 'YOLO', 'ResNet', 'GANs', 'RCNN'
        ]

        thematic_terms = [
            'rock art', 'petroglyph', 'prehistoric', 'engraving', 'pictogram',
            'cave painting', 'rock painting', 'rock engraving',
            'arte rupestre', 'pintura rupestre', 'grabado rupestre', 'petroglifo',
            'art rupestre', 'peinture rupestre', 'gravure rupestre', 'pictogramme', 'pétroglyphe',
            'prehistoric images', 'cave imagery', 'prehistoric symbols',
            'rock drawings', 'rock surface', 'ancient symbols',
            'archaeological images', 'archaeological interpretation',
            'archaeological analysis', 'cultural heritage', 'prehistoric site',
            'archaeoacoustics', 'iconography', 'prehistoric motifs'
        ]

        found_technical = any(term in text for term in technical_terms)
        found_thematic = any(term in text for term in thematic_terms)

        return found_technical and found_thematic

    # Compute keyword signal (no filtering)
    df["keyword_match"] = df["abstract_clean"].apply(contains_keywords)

    print("✅ Keyword signal computed (no early exclusion applied).")
    print(df["keyword_match"].value_counts())

    return df


# Apply
df = compute_keyword_signal(df)

In [ ]:
# =========================
# ABSTRACT SEMANTIC SIMILARITY
# =========================
# Abstract similarity is computed as a continuous score and NOT used for early exclusion

def compute_abstract_similarity(df, threshold=0.60):

    # Device selection
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = "cpu"
        print("GPU not available, using CPU.")

    # Load model
    model = SentenceTransformer('all-mpnet-base-v2', device=device)

    # Research question (multilingual)
    question_en = "What are the digital image processing techniques applied in the analysis and interpretation of rock art images (paintings and engravings), how have they evolved technologically and scientifically over time, what are their main limitations, and what future perspectives can be identified?"
    question_es = "¿Cuáles son las técnicas de procesamiento digital de imágenes aplicadas en el análisis e interpretación del arte rupestre (pinturas y grabados), cómo han evolucionado tecnológica y científicamente a lo largo del tiempo, cuáles son sus principales limitaciones y qué perspectivas futuras pueden identificarse?"
    question_fr = "Quelles sont les techniques de traitement d’images numériques appliquées à l’analyse et à l’interprétation de l’art rupestre (peintures et gravures), comment ont-elles évolué sur les plans technologique et scientifique au fil du temps, quelles sont leurs principales limites, et quelles perspectives futures peuvent être identifiées ?"

    # Multilingual embedding (average)
    question_embedding = torch.mean(torch.stack([
        model.encode(question_en, convert_to_tensor=True),
        model.encode(question_es, convert_to_tensor=True),
        model.encode(question_fr, convert_to_tensor=True)
    ]), dim=0)

    # Ensure clean abstracts
    df["abstract_clean"] = (
        df["abstract"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Encode abstracts
    abstract_embeddings = model.encode(
        df["abstract_clean"].tolist(),
        convert_to_tensor=True,
        batch_size=32,
        show_progress_bar=False
    )

    # Compute cosine similarity
    similarities = util.cos_sim(abstract_embeddings, question_embedding).squeeze()
    df["similarity_abstract"] = similarities.cpu().tolist()

    print("✅ Abstract semantic similarity computed.")
    print(df["similarity_abstract"].describe())

    return df


# Apply
df = compute_abstract_similarity(df, threshold=0.60)

In [ ]:
# Full-text similarity is computed using segmented embeddings and max pooling (no early exclusion)

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer
import pandas as pd
import os
import re
from nltk import sent_tokenize
from tqdm import tqdm

def compute_max_similarity(text, question_embedding, model, tokenizer, token_limit=512):
    sentences = re.split(r'\n\s*\n', text.strip())


    chunks = []
    current_chunk = ""

    for sentence in sentences:
        tokens = tokenizer.encode(current_chunk + " " + sentence, truncation=False, add_special_tokens=False)
        if len(tokens) <= token_limit:
            current_chunk += " " + sentence
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())

    if not chunks:
        return 0.0

    chunk_embeddings = model.encode(chunks, convert_to_tensor=True)
    similarities = util.cos_sim(chunk_embeddings, question_embedding)
    max_sim = similarities.max().item()
    return max_sim


def filter_by_fulltext_similarity_segmentado(df, threshold=0.60):
    # Detecta GPU
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Usando GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = "cpu"
        print("GPU não disponível, usando CPU.")

    # Função auxiliar para carregar texto completo
    def load_full_text(path):
        if pd.isna(path) or not os.path.exists(path):
            return ""
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    # Garante que a coluna full_text existe
    if "full_text" not in df.columns:
        df["full_text"] = df["full_text_path"].apply(load_full_text)
    else:
        df["full_text"] = df["full_text"].fillna("").astype(str)

    # Carrega modelo e tokenizer
    model = SentenceTransformer('all-mpnet-base-v2', device=device)
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")

    # Embedding médio da pergunta
    question_en = "What are the digital image processing techniques applied in the analysis and interpretation of rock art images (paintings and engravings), how have they evolved technologically and scientifically over time, what are their main limitations, and what future perspectives can be identified?"
    question_es = "¿Cuáles son las técnicas de procesamiento digital de imágenes aplicadas en el análisis e interpretación del arte rupestre (pinturas y grabados), cómo han evolucionado tecnológica y científicamente a lo largo del tiempo, cuáles son sus principales limitaciones y qué perspectivas futuras pueden identificarse?"
    question_fr = "Quelles sont les techniques de traitement d’images numériques appliquées à l’analyse et à l’interprétation de l’art rupestre (peintures et gravures), comment ont-elles évolué sur les plans technologique et scientifique au fil du temps, quelles sont leurs principales limites, et quelles perspectives futures peuvent être identifiées ?"

    question_embeddings = torch.mean(torch.stack([
        model.encode(question_en, convert_to_tensor=True),
        model.encode(question_es, convert_to_tensor=True),
        model.encode(question_fr, convert_to_tensor=True)
    ]), dim=0)

    # Processamento com segmentação
    similarities = []
    for text in tqdm(df["full_text"].tolist(), desc="Calculando similaridade segmentada"):
        sim = compute_max_similarity(text, question_embeddings, model, tokenizer)
        similarities.append(sim)

    df["similarity_fulltext"] = similarities
    df["relevanttoquestion_fulltext"] = df["similarity_fulltext"].apply(
        lambda x: "yes" if x >= threshold else "no"
    )

    print("✅ Filtro por similaridade no full text (segmentado) concluído.")
    print(df["relevanttoquestion_fulltext"].value_counts())

    return df

df = filter_by_fulltext_similarity_segmentado(df, threshold=0.60)

-------

*Multi-Criteria Decision Integration*

In [25]:
# 🇬🇧 This block integrates the outputs of the different relevance signals
# (keyword matching, abstract-level similarity, and full-text similarity)
# into a unified multi-criteria decision. For each article, a combined label
# is generated indicating which criteria contributed to its selection.
#
# A rule-based strategy is then applied to determine whether an article is
# recommended for manual review, requiring either (i) at least two positive
# criteria or (ii) a positive full-text similarity signal alone. The results
# are exported to a CSV file containing the subset of articles indicated for review.

def combine_signals_and_export(df, output_csv="data/relevant_articles.csv",
                              threshold_abs=0.60,
                              threshold_full=0.60):

    # Convert signals into boolean criteria
    df["crit_keyword"] = df["keyword_match"]

    df["crit_abstract"] = df["similarity_abstract"] >= threshold_abs
    df["crit_fulltext"] = df["similarity_fulltext"] >= threshold_full

    # Track which criteria contributed
    def combine_decision(row):
        labels = []

        if row["crit_keyword"]:
            labels.append("Keyword")
        if row["crit_abstract"]:
            labels.append("Abstract")
        if row["crit_fulltext"]:
            labels.append("Full Text")

        return " + ".join(labels) if labels else "None"

    df["final_decision_source"] = df.apply(combine_decision, axis=1)

    # Decision rule (f function)
    def should_include(row):
        count = sum([
            row["crit_keyword"],
            row["crit_abstract"],
            row["crit_fulltext"]
        ])

        # Rule:
        # - At least 2 criteria
        # - OR strong full-text signal alone
        if row["crit_fulltext"]:
            return True
        if count >= 2:
            return True
        return False

    df["indicated_for_review"] = df.apply(
        lambda row: "Indicated" if should_include(row) else "Not indicated",
        axis=1
    )

    # Filter selected
    df_relevant = df[df["indicated_for_review"] == "Indicated"]

    # Export
    df_relevant.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print("✅ Final multi-criteria decision completed.")
    print("\n📊 Decision source distribution:")
    print(df["final_decision_source"].value_counts())

    print("\n📊 Final selection:")
    print(df["indicated_for_review"].value_counts())

    return df, df_relevant


# Apply
df, df_relevant = combine_signals_and_export(df)

✅ Final multi-criteria decision completed.

📊 Decision source distribution:
final_decision_source
None                  479
Keyword               147
Keyword + Abstract     30
Abstract                2
Name: count, dtype: int64

📊 Final selection:
indicated_for_review
Not indicated    628
Indicated         30
Name: count, dtype: int64


In [ ]:
len(df_relevant)


*Optional: Copy Relevant Articles*

In [ ]:
# Note: PDF files may not be included due to copyright restrictions.
# This step is optional and will only execute if PDFs are available locally.

# 🇬🇧 This optional block copies the PDF files of articles selected for review
# into categorized folders based on their decision source. This facilitates
# manual inspection and organization of relevant documents.

import os
import shutil

def copy_pdfs_by_decision(df_relevant, pdf_folder, output_base_folder="classified_pdfs"):
    os.makedirs(output_base_folder, exist_ok=True)

    for _, row in df_relevant.iterrows():
        decision = row.get("final_decision_source", "Unknown")
        filename = row.get("filename", "")
        repository = row.get("repository", "")

        # Safety checks
        if not isinstance(repository, str):
            repository = ""
        if not filename or not isinstance(filename, str) or filename.strip() == "":
            print(f"⚠️ Invalid filename, skipping entry.")
            continue

        # Build paths
        src_path = os.path.join(pdf_folder, repository, filename)
        dest_dir = os.path.join(output_base_folder, decision.replace(" + ", "_"))
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, filename)

        # Copy if exists
        if os.path.exists(src_path):
            shutil.copy2(src_path, dest_path)
            print(f"✅ Copied: {filename} -> {dest_dir}")
        else:
            print(f"⚠️ File not found (expected if PDFs are not included): {src_path}")

    print("\n✅ PDF copy process completed.")


# =========================
# EXECUTION (SAFE)
# =========================

pdf_folder_path = "data/pdfs"

if os.path.exists(pdf_folder_path):
    print("📁 PDFs found. Copying relevant files...\n")

    copy_pdfs_by_decision(
        df_relevant,
        pdf_folder=pdf_folder_path,
        output_base_folder="data/pdfs_classificados"
    )

else:
    print("⚠️ PDF folder not found. Skipping copy step (expected in Colab/GitHub).")

---

*Exploratory Analysis of Selected Articles*

*This section presents an exploratory analysis of the articles selected through the automated screening process. The goal is to characterize, both visually and statistically, the set of relevant articles, including the distribution of decision sources and the semantic similarity scores obtained from embedding-based methods.*

In [ ]:
import matplotlib.pyplot as plt

# =========================
# VISUALIZATION
# =========================

# These plots provide an illustrative summary of the final screening decisions
# and the contribution of each criterion in the multi-stage decision function.

# Rename for consistency with paper terminology
df_plot = df.copy()
df_plot["selection_label"] = df_plot["indicated_for_review"].map({
    "Indicated": "Selected",
    "Not indicated": "Not Selected"
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 1. Bar chart: Selected vs Not Selected
selection_counts = df_plot["selection_label"].value_counts()

axes[0].bar(selection_counts.index, selection_counts.values)
axes[0].set_title("Final Screening Decision")
axes[0].set_xlabel("Classification")
axes[0].set_ylabel("Number of Articles")
axes[0].grid(axis="y")


# --- 2. Pie chart: Decision sources (f function interpretation)
decision_counts = df["final_decision_source"].value_counts()

wedges, _ = axes[1].pie(
    decision_counts,
    labels=None,
    startangle=90
)

axes[1].legend(
    wedges,
    [
        f"{label} ({pct:.1f}%)"
        for label, pct in zip(
            decision_counts.index,
            decision_counts / decision_counts.sum() * 100
        )
    ],
    title="Decision Criteria Combination",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

axes[1].set_title("Multi-Criteria Decision Distribution")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# =========================
# SIMILARITY DISTRIBUTION ANALYSIS
# =========================

# These histograms illustrate the distribution of semantic similarity scores
# and support the analysis of the threshold sensitivity discussed in the paper.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Abstract similarity histogram
axes[0].hist(
    df_relevant["similarity_abstract"].dropna(),
    bins=20
)
axes[0].axvline(x=0.60)  # threshold
axes[0].set_title("Abstract-Level Semantic Similarity Distribution")
axes[0].set_xlabel("Similarity Score")
axes[0].set_ylabel("Frequency")


# --- Full-text similarity histogram
axes[1].hist(
    df_relevant["similarity_fulltext"].dropna(),
    bins=20
)
axes[1].axvline(x=0.60)  # threshold
axes[1].set_title("Full-Text Semantic Similarity Distribution")
axes[1].set_xlabel("Similarity Score")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# =========================
# FULL-TEXT SIMILARITY BY DECISION CRITERIA
# =========================

# This plot illustrates how full-text similarity varies across different
# combinations of decision criteria, supporting the analysis of the
# multi-stage decision function described in the paper.

# Prepare data grouped by decision source
groups = df_relevant.groupby("final_decision_source")["similarity_fulltext"].apply(list)

# ✅ Sort groups for consistent ordering (alphabetical)
groups = groups.sort_index()

labels = list(groups.index)
data = list(groups.values)

plt.figure(figsize=(8, 8))

plt.boxplot(data, labels=labels)

# Threshold reference
plt.axhline(y=0.60)

plt.title("Full-Text Similarity by Decision Criteria Combination")
plt.xlabel("Decision Criteria Combination")
plt.ylabel("Similarity Score")

plt.xticks(rotation=20)
plt.grid(axis="y")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================
# CORRELATION ANALYSIS: ABSTRACT vs FULL TEXT
# =========================

# This plot evaluates the relationship between abstract-level and full-text similarity,
# supporting the analysis of their complementary roles in the multi-stage screening process.

# Compute Pearson correlation
correlation = (
    df_relevant[["similarity_abstract", "similarity_fulltext"]]
    .dropna()
    .corr(method="pearson")
    .iloc[0, 1]
)

# Prepare clean data
df_corr = df_relevant.dropna(subset=["similarity_abstract", "similarity_fulltext"])

x = df_corr["similarity_abstract"]
y = df_corr["similarity_fulltext"]

# Fit regression line
coef = np.polyfit(x, y, 1)
poly1d_fn = np.poly1d(coef)

# Plot
plt.figure(figsize=(7, 6))

plt.scatter(x, y)
plt.plot(x, poly1d_fn(x))

plt.title(f"Correlation between Abstract and Full-Text Similarity (Pearson = {correlation:.2f})")
plt.xlabel("Similarity (Abstract)")
plt.ylabel("Similarity (Full Text)")
plt.grid()

plt.tight_layout()
plt.show()


Arquivos de maior concordância (Abstract e Full Text)

In [ ]:

# This ranking is based on the maximum semantic similarity score and is used
# for exploratory analysis only. It does not affect the final inclusion decision,
# which is based on the multi-criteria rule defined in the methodology.

df_relevant = df_relevant.copy()
df_relevant.loc[:, 'similarity_max'] = df_relevant[['similarity_abstract', 'similarity_fulltext']].max(axis=1)


# Selecionar top 30 com maior similaridade geral
top30_combined = (
    df_relevant.sort_values(by='similarity_max', ascending=False)
    .head(30)
    .reset_index(drop=True)
)

# Adicionar Rank
top30_combined['Rank'] = top30_combined.index + 1

# Renomear colunas para exibição mais amigável
top30_combined.rename(columns={
    'filename': 'Filename',
    'similarity_fulltext': 'Similarity (Full Text)',
    'similarity_abstract': 'Similarity (Abstract)',
    'final_decision_source': 'Decision Source'
}, inplace=True)

# Organizar colunas
cols_to_show = [
    'Rank',
    'Filename',
    'Similarity (Abstract)',
    'Similarity (Full Text)',
    'Decision Source'
]

# Exibir tabela
display(top30_combined[cols_to_show])
